In [54]:
# for working part
import numpy as np
import pandas as pd



#  for ml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler, PowerTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score,recall_score,f1_score,roc_auc_score
from sklearn.model_selection import RandomizedSearchCV,cross_val_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline


# for saving final selected models and thresholds details
import joblib
import json
from pathlib import Path

In [2]:
# reading the parquet file created in notebook 2

lead_conversion_data = pd.read_parquet("C:\\Users\\hp5cd\\OneDrive\\Desktop\\Python\\lead-conversion-prediction\\data\\processed_data.parquet")

In [3]:
# hume neeche model train krne mein errors aaye toh humne ye kiya hai again aur phir se re run krne ja rhe hai codes

lead_conversion_data['inbound_outbound_ratio'] = lead_conversion_data['inbound_outbound_ratio'].replace([np.inf,-np.inf],np.nan).fillna(0)

lead_conversion_data.fillna(0,inplace=True)

lead_conversion_data['profile'] = lead_conversion_data['profile'].replace(0,"unknown_profile")

In [4]:
# extracting important columns for this model
# as this will be used when creating leads we will have columns only that can be there after creating leads (x1)
# and when assigned we will have the columns that are in x2 


x1 = lead_conversion_data[['lead_source']]
x2 = lead_conversion_data[['lead_source','owner','assigned_month','assigned_year']]
x3 = lead_conversion_data[['owner', 'lead_source', 'profile', 'total_duration', 'call_count',
       'distinct_call_days', 'connected_call_count', 'missed_call_count',
       'inbound_call_count', 'outbound_call_count',
       'assigned_month', 'assigned_year', 'followup_done', 'average_duration',
       'connection_rate', 'miss_rate', 'average_call_per_day',
       'average_duration_per_day', 'inbound_outbound_ratio',
       'time_taken_for_first_touch', 'call_span_days', 'call_frequency']]


y = lead_conversion_data['converted']


In [5]:
# splitting again for the three models

x1_train,x1_test,y1_train,y1_test = train_test_split(x1,y,random_state=42,stratify=y,test_size=0.2)

x2_train,x2_test,y2_train,y2_test = train_test_split(x2,y,random_state=42,stratify=y,test_size=0.2)

x3_train,x3_test,y3_train,y3_test = train_test_split(x3,y,random_state=42,stratify=y,test_size=0.2)

In [6]:
#  columns seperation

categorical_x1 = ['lead_source']

categorical_x2 = ['lead_source','owner','assigned_month','assigned_year']

categorical_x3 = ['owner', 'lead_source', 'profile','assigned_month',
                   'assigned_year']

numerical_x3 = ['total_duration', 'call_count','distinct_call_days',
                 'connected_call_count', 'missed_call_count',
                 'inbound_call_count', 'outbound_call_count','average_duration',
                 'connection_rate', 'miss_rate', 'average_call_per_day',
                 'average_duration_per_day', 'inbound_outbound_ratio',
                 'time_taken_for_first_touch', 'call_span_days', 'call_frequency']

binary_x3 = ['followup_done']

In [ ]:
# creating a dictionary so that i dont need to write multiple times datasets name

dataset_config = {"Source" :{"x_train":x1_train,
                             "x_test":x1_test,
                             "y_train":y1_train,
                             "y_test":y1_test,
                             
                             'cat': categorical_x1,
                             'num':None,
                             'binary':None},

                "Assigned":{"x_train":x2_train,
                             "x_test":x2_test,
                             "y_train":y2_train,
                             "y_test":y2_test,
                             
                             'cat': categorical_x2,
                             'num':None,
                             'binary':None},
                             
                "Dynamic":{"x_train":x3_train,
                             "x_test":x3_test,
                             "y_train":y3_train,
                             "y_test":y3_test,
                             
                             'cat': categorical_x3,
                             'num':numerical_x3,
                             'binary':binary_x3}}



In [ ]:
# calculating imbalance ratio for Xg boost hyperparameter

def scale_pos_weight(y_train):
    return y_train.value_counts()[0] / y_train.value_counts()[1]

In [ ]:
# creating a preprocessor function as we have to do this task repeatedly


def preprocessor(categorical_cols,numerical_cols = None, binary_cols = None, transformation=None):
    transformers = []

    if categorical_cols:
        transformers.append(('cat',OneHotEncoder(handle_unknown='ignore'), categorical_cols))


    if numerical_cols:
        if transformation is None:
            num_pipeline = Pipeline([('scaler',StandardScaler())])
        elif transformation == 'power':
            num_pipeline = Pipeline([('power',PowerTransformer(method='yeo-johnson')),
                                     ('scaler',StandardScaler())])
        
        transformers.append(('num',num_pipeline,numerical_cols))



    if binary_cols:
        transformers.append(('binary','passthrough',binary_cols))

    preprocessor = ColumnTransformer(transformers=transformers)


    return preprocessor

In [ ]:
# creating a modelss function as we have to do this task repeatedly

def modelss(class_weight = None, scale_pos_weight = 1, rf_estimators = 5, rf_max_depth = None,
            xgb_estimators = 50, xgb_max_depth = 5, xgb_lr = 0.1):
    
    models = {'LR':LogisticRegression(max_iter=1000,class_weight=class_weight),
              
              'RF':RandomForestClassifier(n_estimators=rf_estimators,max_depth=rf_max_depth,n_jobs=-1,
                                              random_state=42,class_weight=class_weight),

                'XGB':XGBClassifier(n_estimators=xgb_estimators,max_depth=xgb_max_depth,learning_rate=xgb_lr,
                                      n_jobs=-1,random_state=42,scale_pos_weight = scale_pos_weight)}
    

    return models

In [ ]:
# this function basically merges preprocess and model creation function

def get_preprocessor_and_models(data,transformation = None, class_weight= None):

    preprocess = preprocessor(categorical_cols=data['cat'],
                              numerical_cols=data['num'],
                              binary_cols=data['binary'],
                              transformation = transformation)
    
    ratio = scale_pos_weight(data['y_train'])

    models = modelss(class_weight=class_weight,scale_pos_weight=ratio)


    return preprocess, models

In [ ]:
# function for training models based on merge function created above

def train_models(preprocessor,models,x_train,y_train,use_smote = False):
    trained_models = {}

    for name,model in models.items():
        if use_smote == True:
            pipeline = Pipeline([('preprocessor',preprocessor),
                                 ('smote',SMOTE(random_state=42)),
                                 ('model',model)])
        else:
            pipeline = Pipeline([('preprocessor',preprocessor),
                                 ('model',model)])
            
        pipeline.fit(x_train,y_train)
        trained_models[name] = pipeline

    return trained_models

In [13]:
# creating a model evaluatation function as we have to do this task repeatedly



def evaluate (trained_models, x_test,y_test):
    results=[]

    for name,pipeline in trained_models.items():
        y_pred = pipeline.predict(x_test)
        y_pred_prob = pipeline.predict_proba(x_test)[:,1]

        results.append({
            'name': name,
            'accuracy_score': accuracy_score(y_test,y_pred),
            "confusion_matrix": confusion_matrix(y_test,y_pred),
            "precision_score": precision_score(y_test,y_pred),
            "recall_score": recall_score(y_test,y_pred),
            "f1_score": f1_score(y_test,y_pred),
            "roc_auc_score": roc_auc_score(y_test,y_pred_prob)
        })


    return pd.DataFrame(results)

In [ ]:
# experiment function, so i have not to write same code multiple times when i am expermenting the models


def run_experiment(datasets,transformation = None,class_weight = None,use_smote = False,label='Experiment'):
    all_results = {}
    all_models = {}

    for dataset_name,data in datasets.items():
        print("-"*40)
        print(label,": ",dataset_name)
        print("-"*40)

        preprocess, models = get_preprocessor_and_models(data,
                                                         transformation=transformation,
                                                         class_weight=class_weight)
        
        trained = train_models(preprocess,models,data['x_train'],data['y_train'],use_smote=use_smote)

        results = evaluate(trained,data['x_test'],data['y_test'])


        all_results[dataset_name] = results

        all_models[dataset_name] = trained

        print(results)
        print()

    return all_results,all_models


In [ ]:
# function to tune models on hyperparameters

def tune_models(datasets,param_grids,transformation=None,class_weight= None,
                n_iter=5,cv=2,scoring='roc_auc',random_state = 42):
    
    best_models = {}
    tuning_summary = {}

    for dataset_name,data in datasets.items():
        print("-"*40)
        print("tuning: ",dataset_name)
        print("-"*40)

        preprocess,models = get_preprocessor_and_models(data,transformation=transformation,class_weight=class_weight)

        best_models[dataset_name] = {}
        tuning_summary[dataset_name] = {}

        for name,model in models.items():
            if name not in param_grids:
                continue
            pipeline = Pipeline([('preprocessor',preprocess),('model',model)])

            search = RandomizedSearchCV(estimator=pipeline,
                                        param_distributions=param_grids[name],
                                        n_iter=n_iter,
                                        cv = cv,
                                        scoring=scoring,
                                        random_state=random_state,
                                        n_jobs=-1)
            search.fit(data['x_train'],data['y_train'])

            best_models[dataset_name][name] = search.best_estimator_
            tuning_summary[dataset_name][name] = {'best_params':search.best_params_,
                                                  'best_score':search.best_score_}
            
            print(name,"best cv score: ",search.best_score_)

    return best_models,tuning_summary



In [ ]:
# defining hyperparameters options

param_grids = {
    'LR': {
        'model__C': [0.01, 0.1, 1, 10],
        'model__penalty': ['l2'],
        'model__solver': ['lbfgs', 'liblinear']
    },
    'RF': {
        'model__n_estimators': [ 10, 20],
        'model__max_depth': [ 5, 10, 15],
        'model__min_samples_split': [2, 5, 10]
    },
    'XGB': {
        'model__n_estimators': [50, 100, 200],
        'model__max_depth': [3, 5, 7],
        'model__learning_rate': [0.01, 0.05, 0.1, 0.2]
    }
}

In [ ]:
# function for threshold tuning as baseline models take 0.50 probability and we can increase or decrease to improve the models

def threshold_tuning(trained_models,x_test,y_test,thresholds=None):
    if thresholds is None:
        thresholds = [0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70]

    rows = []
    for name,pipeline in trained_models.items():
        y_pred_prob = pipeline.predict_proba(x_test)[:,1]

        for t in thresholds:
            y_pred = (y_pred_prob >= t).astype(int)
            rows.append({'model':name,
                         'threshold':t,
                         'precision_score':precision_score(y_test,y_pred,zero_division=0),
                         'recall_score': recall_score(y_test,y_pred,zero_division=0),
                         'f1_score':f1_score(y_test,y_pred,zero_division=0)})
            
    return pd.DataFrame(rows)



In [ ]:
# def cross_validate_models(datasets,transformation = None,class_weight= None,cv=2,scoring='roc_auc'):
#     cv_results = []
#     for dataset_name,data in datasets.items():
#         preprocess,models = get_preprocessor_and_models(data,transformation= transformation,class_weight=class_weight)

#         for name,model in models.items():
#             pipeline = Pipeline([('preprocessor',preprocess),('model',model)])

#             scores = cross_val_score(pipeline,data['x_train'],data['y_train'],
#                                      cv=cv, scoring=scoring,n_jobs=-1)
            
#             cv_results.append({'dataset':dataset_name,
#                                'model':name,
#                                'mean_score':scores.mean(),
#                                'std_score':scores.std()})
            

#     return pd.DataFrame(cv_results)

In [ ]:
# function to basically compare all the experiments we will do later in the notebook


def compare_results(results_dict):
    combined = []

    for experiment_name, dataset_results in results_dict.items():
        for dataset_name, df in dataset_results.items():
            temp = df.copy()
            temp['experiment'] = experiment_name
            temp['dataset'] = dataset_name
            combined.append(temp)

    final_df = pd.concat(combined, ignore_index=True)

    cols = ['experiment', 'dataset', 'name', 'accuracy_score', 'precision_score',
            'recall_score', 'f1_score', 'roc_auc_score']
    cols = [c for c in cols if c in final_df.columns]

    return final_df[cols]

In [ ]:
# training baseling models of the three stages

baseline_results, baseline_models = run_experiment(dataset_config)

----------------------------------------
Experiment :  Source
----------------------------------------


c:\Users\hp5cd\OneDrive\Desktop\Python\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hp5cd\OneDrive\Desktop\Python\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


  name  accuracy_score                  confusion_matrix  precision_score  \
0   LR        0.675114          [[94516, 0], [45484, 0]]          0.00000   
1   RF        0.675114          [[94516, 0], [45484, 0]]          0.00000   
2  XGB        0.602557  [[56950, 37566], [18076, 27408]]          0.42183   

   recall_score  f1_score  roc_auc_score  
0      0.000000  0.000000       0.637926  
1      0.000000  0.000000       0.637926  
2      0.602586  0.496261       0.637926  

----------------------------------------
Experiment :  Assigned
----------------------------------------
  name  accuracy_score                  confusion_matrix  precision_score  \
0   LR        0.678279    [[91174, 3342], [41699, 3785]]         0.531079   
1   RF        0.664807   [[83627, 10889], [36038, 9446]]         0.464519   
2  XGB        0.607857  [[57400, 37116], [17784, 27700]]         0.427364   

   recall_score  f1_score  roc_auc_score  
0      0.083216  0.143886       0.644828  
1      0.207677  0

In [ ]:
# experiment 1 -- trying distribution transformation and training the model, choosed yeo-jhonson as was not sure which to choose log or any other

dynamic_datset = {'Dynamic':dataset_config['Dynamic']}

power_results,power_models = run_experiment(dynamic_datset,transformation='power')

----------------------------------------
Experiment :  Dynamic
----------------------------------------
  name  accuracy_score                  confusion_matrix  precision_score  \
0   LR        0.724457  [[78377, 16139], [22437, 23047]]         0.588144   
1   RF        0.681500  [[74227, 20289], [24301, 21183]]         0.510778   
2  XGB        0.665386   [[54442, 40074], [6772, 38712]]         0.491356   

   recall_score  f1_score  roc_auc_score  
0      0.506706  0.544396       0.790352  
1      0.465724  0.487212       0.726088  
2      0.851112  0.623030       0.788262  



In [ ]:
# experiment 2 -- changing the class_weight and training the models and evaluating, in this we give importance to one class more than the other

balanced_results,balanced_models = run_experiment(dataset_config,class_weight='balanced')

----------------------------------------
Experiment :  Source
----------------------------------------
  name  accuracy_score                  confusion_matrix  precision_score  \
0   LR        0.602557  [[56950, 37566], [18076, 27408]]          0.42183   
1   RF        0.602557  [[56950, 37566], [18076, 27408]]          0.42183   
2  XGB        0.602557  [[56950, 37566], [18076, 27408]]          0.42183   

   recall_score  f1_score  roc_auc_score  
0      0.602586  0.496261       0.637926  
1      0.602586  0.496261       0.637926  
2      0.602586  0.496261       0.637926  

----------------------------------------
Experiment :  Assigned
----------------------------------------
  name  accuracy_score                  confusion_matrix  precision_score  \
0   LR        0.609864  [[57647, 36869], [17750, 27734]]         0.429299   
1   RF        0.587379  [[55876, 38640], [19127, 26357]]         0.405511   
2  XGB        0.607857  [[57400, 37116], [17784, 27700]]         0.427364   

 

In [ ]:
# experiment 3 -- hyperparameter tuning

best_models,tuning_summary = tune_models(dataset_config,param_grids)

tuned_results = {name:evaluate(models,dataset_config[name]['x_test'],dataset_config[name]['y_test'])
                 for name,models in best_models.items()}

----------------------------------------
tuning:  Source
----------------------------------------


c:\Users\hp5cd\OneDrive\Desktop\Python\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


LR best cv score:  0.6408494945248723
RF best cv score:  0.6408494945248723
XGB best cv score:  0.6408494945248723
----------------------------------------
tuning:  Assigned
----------------------------------------


c:\Users\hp5cd\OneDrive\Desktop\Python\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


LR best cv score:  0.6466686626868443
RF best cv score:  0.6411848325210057
XGB best cv score:  0.6460935650142541
----------------------------------------
tuning:  Dynamic
----------------------------------------


c:\Users\hp5cd\OneDrive\Desktop\Python\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


LR best cv score:  0.7899632836652712
RF best cv score:  0.7674870234059551
XGB best cv score:  0.7892321724719322


c:\Users\hp5cd\OneDrive\Desktop\Python\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hp5cd\OneDrive\Desktop\Python\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hp5cd\OneDrive\Desktop\Python\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.sh

In [ ]:
# comparing all experiments
# we did not do SMOTE, because the data is not that much imbalanced
# we did not do cross validation, as we did that in hyperparameter tuning experiment


all_experiment_results = {
    'Baseline': baseline_results,
    'Power': power_results,
    'Balanced': balanced_results,
    'Tuned': tuned_results
}

final_comparison = compare_results(all_experiment_results)
final_comparison.sort_values('roc_auc_score', ascending=False)

,experiment,dataset,name,accuracy_score,precision_score,recall_score,f1_score,roc_auc_score
6,Baseline,Dynamic,LR,0.724336,0.588104,0.505650,0.543769,0.790366
9,Power,Dynamic,LR,0.724457,0.588144,0.506706,0.544396,0.790352
27,Tuned,Dynamic,LR,0.724000,0.587983,0.502792,0.542061,0.790350
18,Balanced,Dynamic,LR,0.672900,0.497972,0.836932,0.624418,0.790344
29,Tuned,Dynamic,XGB,0.670871,0.496135,0.838119,0.623300,0.789334
11,Power,Dynamic,XGB,0.665386,0.491356,0.851112,0.623030,0.788262
20,Balanced,Dynamic,XGB,0.665386,0.491356,0.851112,0.623030,0.788262
8,Baseline,Dynamic,XGB,0.665386,0.491356,0.851112,0.623030,0.788262
28,Tuned,Dynamic,RF,0.706079,0.601986,0.281286,0.383415,0.767106
19,Balanced,Dynamic,RF,0.669014,0.492611,0.625846,0.551293,0.734744


In [ ]:
# MODEL SELECTION

## i will first rank the models according to roc_auc(measures model's ranking ability) 
## then i would pick based on f1 score of the model
##  as the business requires good value of recall(to capture converted leads) but precision(to avoid wasting sales effort) also


# first model ---- source
# in this choosing BALANCED LR because roc_auc is same for all but in this f1 score is good, 
# also this same is in RF AND XGB but why to process more for same results that we can get easily

# second model ---- assigned
# in this also choosing BALANCED LR because roc_auc high and f1 score is also max


# third model ---- dynamic
# i am choosing BALANCED LR because roc auc score is almost max but the f1 score is high, so we can choose this



final_models = {'Source':{'LR': balanced_models['Source']['LR']},
                'Assigned':{'LR': balanced_models['Assigned']['LR']},
                'Dynamic':{'LR': balanced_models['Dynamic']['LR']}}


In [ ]:
# now we are trying threshold changes on selected models above to check if there is some improvement or not
# we should always do threshold tunning after model selection

threshold_results = {}

for dataset_name, data in dataset_config.items():
    threshold_results[dataset_name] = threshold_tuning(

        final_models[dataset_name],

        data['x_test'],

        data['y_test'])
    
    print(dataset_name)

    print(threshold_results[dataset_name].sort_values('f1_score',ascending = False))
    print()

Source
  model  threshold  precision_score  recall_score  f1_score
2    LR       0.40         0.400958      0.734170  0.518658
3    LR       0.45         0.400958      0.734170  0.518658
1    LR       0.35         0.348184      0.911705  0.503919
4    LR       0.50         0.421830      0.602586  0.496261
0    LR       0.30         0.324886      1.000000  0.490436
5    LR       0.55         0.461017      0.470737  0.465826
6    LR       0.60         0.461017      0.470737  0.465826
7    LR       0.65         0.480830      0.339416  0.397933
8    LR       0.70         0.000000      0.000000  0.000000

Assigned
  model  threshold  precision_score  recall_score  f1_score
3    LR       0.45         0.405602      0.720121  0.518925
2    LR       0.40         0.386902      0.781945  0.517666
1    LR       0.35         0.358111      0.879232  0.508934
4    LR       0.50         0.429299      0.609753  0.503856
0    LR       0.30         0.331318      0.980718  0.495306
5    LR       0.55     

In [ ]:
# THRESHOLD SELECTION

## i would first take high f1 score, then if tie i would take higher recall


# first model ---- source
# choosing threshold = 0.40, because the f1 is high, recall is high, precision is ok, 
# now it is same with 0.45 which suggests that no data lies in 0.40 to 0.45 but in test we can get that so we are taking a safer threshold value

# second model ---- assigned
# choosing threshold = 0.40, because the f1 is almost same with max and recall is higher which is important for business perspective(potential converted leads should not miss)

# third model ---- dynamic
# choosing threshold = 0.45, because the f1 difference with max is negligible and increase of recall is there and precision is also not decreasing that much



final_thresholds = {"Source":0.40,
                    "Assigned":0.40,
                    "Dynamic":0.45}


In [ ]:
# saving the final models selected in the respective folder

model_path = Path("C:\\Users\\hp5cd\\OneDrive\\Desktop\\Python\\lead-conversion-prediction\\models")

for dataset_name,model in final_models.items():
    file_name = dataset_name.lower() + "_model.joblib"

    joblib.dump(model,model_path/file_name)

    print(file_name,"model saved")

source_model.joblib model saved
assigned_model.joblib model saved
dynamic_model.joblib model saved


In [ ]:
# saving the final thresholds selected into json

with open (model_path/"thresholds.json","w") as f:
    json.dump(final_thresholds,f,indent=4)